In [1]:

import numpy as np
import pandas as pd
# ------------------------------
# 1. PARAMETERS
# ------------------------------
np.random.seed(42)
n_ships = 200
years = [1,2,3,4,5]

# Ship type probabilities
ship_types = ['offshore','tanker','container','bulk','lpg']
ship_probs = [0.90, 0.05, 0.01, 0.03, 0.01]

# K4K usage probability by ship type
prob_k4k = {'offshore':0.95,'tanker':0.40,'container':0.05,'bulk':0.20,'lpg':0.30}

# DP operations probability
prob_dp = {'offshore':0.92,'tanker':0.04,'container':0.01,'bulk':0.01,'lpg':0.02}

# Ship-type structural effects
sector_effects = {'offshore':0.80,'tanker':0.50,'container':0.20,'bulk':0.40,'lpg':0.45}

# GLM / Negative Binomial Coefficients
β0 = -2.0
β_k4k = -0.25
β_age = 0.02
β_ism = -0.035
β_crew = -0.05
β_dp = 0.40
β_weather = 1.20
β_failed = 0.70

# ------------------------------
# 2. CREATE SHIP MASTER TABLE
# ------------------------------
ship_master = pd.DataFrame({
    "ship_id": np.arange(1, n_ships+1),
    "ship_type": np.random.choice(ship_types, size=n_ships, p=ship_probs),
})

# ------------------------------
# 3. EXPAND TO PANEL DATA (5 years)
# ------------------------------
records = []

for _, row in ship_master.iterrows():

    s_id = row.ship_id
    s_type = row.ship_type

    for yr in years:

        # Generate variables
        age = np.random.randint(1,30)
        ism_score = np.random.uniform(50,95)
        crew_exp = np.random.uniform(1,25)
        weather_risk = np.random.choice([0,1], p=[0.85,0.15])
        failed_audit = np.random.choice([0,1], p=[0.90,0.10])
        k4k = np.random.choice([0,1], p=[1-prob_k4k[s_type], prob_k4k[s_type]])
        dp = np.random.choice([0,1], p=[1-prob_dp[s_type], prob_dp[s_type]])

        # Log mean accident rate
        log_mu = (
            β0
            + β_k4k * k4k
            + β_age * age
            + β_ism * ism_score
            + β_crew * crew_exp
            + β_dp * dp
            + β_weather * weather_risk
            + β_failed * failed_audit
            + sector_effects[s_type]
        )

        mu = np.exp(log_mu)

        # Generate accidents (negative binomial)
        accidents = np.random.negative_binomial(n=1, p=1/(1+mu))

        records.append([
            s_id, yr, s_type, age, ism_score, crew_exp, k4k, dp,
            weather_risk, failed_audit, mu, accidents
        ])

# ------------------------------
# 4. FINAL DATASET
# ------------------------------
df = pd.DataFrame(records, columns=[
    "ship_id","year","ship_type","age","ism_score","crew_exp","k4k","dp_ops",
    "weather_risk","failed_audit","mu","accidents"
])

df.head()
df.to_csv("synthetic_maritime_dataset.csv", index=False)

df



,ship_id,year,ship_type,age,ism_score,crew_exp,k4k,dp_ops,weather_risk,failed_audit,mu,accidents
0,1,1,offshore,24,88.291780,23.455240,1,1,0,0,0.007963,0
1,1,2,offshore,18,57.236362,14.169611,1,1,0,0,0.033315,0
2,1,3,offshore,20,66.491095,18.860093,1,1,0,0,0.019837,0
3,1,4,offshore,6,94.050179,12.681812,1,1,1,0,0.025840,0
4,1,5,offshore,28,82.510345,7.738537,1,1,0,0,0.023173,0
...,...,...,...,...,...,...,...,...,...,...,...,...
995,200,1,offshore,9,94.193482,10.386214,1,1,0,0,0.009223,0
996,200,2,offshore,2,83.033309,1.738604,1,1,0,0,0.018259,0
997,200,3,offshore,22,84.978301,20.970274,1,1,0,0,0.009728,0
998,200,4,offshore,21,86.270527,7.076980,1,0,0,0,0.012237,0
